# DATALUS POC Quickstart: Pipeline Generativo SIH-SUS

Este notebook demonstra o framework **DATALUS** para geração de dados tabulares sintéticos.
Utilizamos uma amostra real de dados de saúde brasileiros (SIH-SUS) para demonstrar todo o pipeline:
da ingestão de dados brutos à exportação ONNX para o edge, incluindo auditoria e operações generativas.

O **DATALUS** foi projetado para conjuntos de dados governamentais heterogêneos e de alta dimensão, 
garantindo privacidade (alinhamento com a LGPD) e utilidade por meio de modelos de difusão avançados.

## 1. Configuração do Ambiente
Detectando o ambiente e instalando dependências se necessário.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_poc'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_poc'
    return './datalus_poc'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

INSTALLED = Path(WORKING_DIR) / '.installed'

if not INSTALLED.exists():
    if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        !git clone --depth 1 https://github.com/emanuellcs/datalus.git "{WORKING_DIR}/source" 2>/dev/null || true
        %cd "{WORKING_DIR}/source"
        !pip install --quiet -e '.[dev]' pysus polars matplotlib nest_asyncio 2>&1 | tail -5
        sys.path.insert(0, f'{WORKING_DIR}/source')
        INSTALLED.touch()
        print()
        print('=' * 60)
        print('  INSTALAÇÃO CONCLUÍDA')
        print('  Clique em "RESTART RUNTIME" no aviso acima')
        print('  ou vá em Ambiente de execução > Reiniciar e executar tudo')
        print('=' * 60)
        os._exit(0)
    else:
        INSTALLED.touch()
        print('Ambiente local. Dependências consideradas instaladas.')
else:
    src = Path(WORKING_DIR) / 'source'
    if src.is_dir():
        sys.path.insert(0, str(src))
    print('Dependências já instaladas.')


## 2. Ingestão de Dados
Buscamos dados públicos de saúde brasileiros através de múltiplas estratégias de fallback.
A fonte primária são os registros clínicos SIH-SUS (grupo `RD`) contendo
a coluna `MORTE` (indicador de óbito). Se o SIH estiver indisponível, tentamos
SIM (mortalidade), SINAN (notificações de doenças), FTP direto do DATASUS,
ou uma amostra embutida para garantir que o pipeline sempre tenha dados para processar.
Esta abordagem agnóstica a dataset funciona com qualquer setor governamental brasileiro.

In [ ]:
from pysus import sih, sim, sinan
import polars as pl
import numpy as np
from datetime import datetime
from pathlib import Path
import urllib.request
import asyncio

print('Downloading clinical data from DATASUS...')

STATES = ['SP', 'RJ', 'MG', 'PR', 'RS', 'BA', 'DF']

def fmt_tier(label, state, year, month, count):
    return f'Tier {label}: {state}/{year}/{month} ({count} records)'

def try_pysus():
    now = datetime.now()
    sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
    for state in STATES:
        for y in range(sy, sy - 2, -1):
            for m in range(sm if y == sy else 12, 0, -1):
                try:
                    df = sih(state=state, year=y, month=[m], group='RD', as_dataframe=True)
                    if df is not None and len(df) > 0 and 'MORTE' in df.columns:
                        print(fmt_tier('1: SIH-RD', state, y, m, len(df)))
                        return pl.from_pandas(df)
                except Exception:
                    continue
        for y in range(sy, sy - 2, -1):
            try:
                df = sim(state=state, year=y, as_dataframe=True)
                if df is not None and len(df) > 0:
                    print(fmt_tier('2: SIM', state, y, 'all', len(df)))
                    df['MORTE'] = 1
                    return pl.from_pandas(df)
            except Exception:
                continue
    for y in range(sy, sy - 2, -1):
        try:
            df = sinan(disease='deng', year=y, as_dataframe=True)
            if df is not None and len(df) > 0:
                print(fmt_tier('3: SINAN-deng', 'BR', y, 'all', len(df)))
                if 'MORTE' not in df.columns:
                    df['MORTE'] = 0
                return pl.from_pandas(df)
        except Exception:
            continue
    return None

def try_direct_ftp():
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        return None
    from pysus.api.extensions import DBC
    now = datetime.now()
    sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
    base = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/199301_/Dados/'
    for state in ['SP', 'RJ', 'MG']:
        for y in range(sy, sy - 2, -1):
            for m in range(sm if y == sy else 12, 0, -1):
                fname = f'RD{state}{y % 100:02d}{m:02d}.dbc'
                try:
                    with urllib.request.urlopen(base + fname, timeout=30) as r:
                        dbc_path = Path('/tmp') / fname
                        dbc_path.write_bytes(r.read())
                    df = asyncio.run(DBC(path=dbc_path).load())
                    if df is not None and len(df) > 0:
                        print(fmt_tier('4: FTP direct', state, y, m, len(df)))
                        return pl.from_pandas(df)
                except Exception:
                    continue
    return None

def generate_sample():
    print('Tier 5: Generated demo sample (real data unavailable)')
    print('NOTE: This is not real data — results are for pipeline demo only.')
    np.random.seed(42)
    n = 1000
    return pl.DataFrame({
        'MORTE': np.random.choice([0, 1], n, p=[0.92, 0.08]).astype(np.int64),
        'IDADE': np.random.randint(0, 100, n).astype(np.float64),
        'SEXO': np.random.choice(['0', '1'], n),
        'DIAS_PERMANENCIA': np.clip(np.random.poisson(5, n), 0, 100).astype(np.float64),
        'VALOR_TOTAL': np.round(np.random.lognormal(7, 1.5, n), 2),
        'PROCEDIMENTO_PRINCIPAL': np.random.choice(['AB', 'CD', 'EF', 'GH', 'IJ'], n),
        'MUNICIPIO_MOV': np.random.choice(['3550308', '3304557', '3106200'], n),
    })

df_polars = try_pysus()
if df_polars is None:
    df_polars = try_direct_ftp()
if df_polars is None:
    df_polars = generate_sample()

df_polars = df_polars.with_columns(
    pl.col('MORTE').cast(pl.Int64).fill_null(0)
)

cols_to_keep = [
    c for c in df_polars.columns
    if not c.startswith(('N_AIH', 'IDENT', 'CNS', 'CPF', 'CNPJ'))
]
df_polars = df_polars.select(
    ['MORTE'] + [c for c in cols_to_keep if c != 'MORTE']
).head(10000)

raw_path = f'{WORKING_DIR}/raw_sih.parquet'
df_polars.write_parquet(raw_path)
print(f'Data saved to {raw_path} | Shape: {df_polars.shape}')
print(f'Columns ({len(df_polars.columns)}): {df_polars.columns[:8]}...')

## 3. Ingestão com a CLI do DATALUS
O comando `ingest` infere o esquema, detecta delimitadores e prepara os dados para modelagem.

In [ ]:
!datalus ingest {WORKING_DIR}/raw_sih.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE

## 4. Treinando o Modelo de Difusão
Treinamos o denoiser MLP residual. Para este POC, usamos um número muito baixo de épocas.

In [ ]:
!datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 3 --batch-size 1024 --gpu 0

## 5. Geração Ab-initio (Amostragem)
Gerando registros sintéticos inteiramente novos a partir da distribuição aprendida.

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 5000

## 6. Aumento de Dados (Augmentation)
Anexando linhas sintéticas a um pequeno conjunto de dados existente.

In [ ]:
synthetic = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')
seed_for_augment = synthetic.sample(n=1000)
seed_for_augment.write_parquet(f'{WORKING_DIR}/seed_for_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_for_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000

## 7. Balanceamento de Classe Minoritária
Gerando registros para balancear classes de destino específicas (ex: mortalidade).

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 2000}'

## 8. Inpainting Tabular (RePaint)
Preenchendo valores faltantes (nulos) preservando os campos observados.

In [ ]:
processed = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
# Criar valores ausentes na coluna 'IDADE' para demonstração
mask = np.random.rand(100) > 0.5
incomplete = processed.head(100).with_columns(
    pl.when(pl.lit(mask)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet

## 9. Modificação Contrafactual
Aplicando intervenções (ex: alterando 'SEXO') e regenerando campos compatíveis.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}'

## 10. Auditoria Autônoma de Privacidade e Utilidade
Avaliando métricas de DCR e razões de utilidade TSTR/TRTR.

In [ ]:
!datalus audit {WORKING_DIR}/processed.parquet {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/schema_config.json {WORKING_DIR}/audit_report.json --target-column MORTE --mia-mode ci_lite

## 11. Visualizando Resultados da Auditoria


In [ ]:
import json
import matplotlib.pyplot as plt

with open(f'{WORKING_DIR}/audit_report.json', 'r') as f:
    report = json.load(f)

print(json.dumps(report, indent=2))

# Plotar Utilidade (TSTR vs TRTR)
if 'utility' in report and 'target_class_probs' in report['utility']:
    probs = report['utility']['target_class_probs']
    labels = list(probs.keys())
    real_probs = [probs[l]['real'] for l in labels]
    synth_probs = [probs[l]['synthetic'] for l in labels]
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots()
    ax.bar(x - width/2, real_probs, width, label='Real')
    ax.bar(x + width/2, synth_probs, width, label='Synthetic')
    
    ax.set_ylabel('Probability')
    ax.set_title('Target Class Distribution Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    plt.show()

## 12. Exportando para ONNX Edge
Exportando pesos para inferência local no navegador ou edge.

In [ ]:
!datalus export-onnx {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/artifacts --quantize

## 13. Servindo e Implantação Web
Os artefatos podem ser servidos via FastAPI:
`datalus serve {WORKING_DIR}/artifacts --host 0.0.0.0 --port 8000`

Isso permite que o frontend React WASM realize inferência localmente no navegador do usuário.